In [1]:
import chromadb

In [2]:
chroma_client = chromadb.Client()

In [3]:
collection = chroma_client.get_or_create_collection(name="my_collection-test")

In [4]:
import polars as p1

articles = p1.read_csv("D:\\AI-Experiments\\chromadb\\defect_dataset.csv", encoding='ISO-8859-1')

# Remove duplicate descriptions, keeping the first occurrence
articles = articles.unique(subset=['Description'], keep='first')

In [5]:
#!pip install openai

In [6]:
# Import the SentenceTransformer library
#from sentence_transformers import SentenceTransformer

# Load a pre-trained SentenceTransformer model for generating embeddings
#model = SentenceTransformer('all-MiniLM-L6-v2')

# Define a function to generate embeddings for a given text
#def embedding_function(text):
#    return model.encode(text)


In [7]:
#N = 50
#articles = articles[:N]
# Generate a list of IDs matching the length of the articles
ids_list = [f"id{i}" for i in range(len(articles['Description']))]
#ids_list

In [8]:
articles_list = articles['Description'].to_list() 
#articles_list

In [9]:
from chromadb.utils.embedding_functions.ollama_embedding_function import (
    OllamaEmbeddingFunction,
)

ollama_ef = OllamaEmbeddingFunction(
    url="http://localhost:11434",
    model_name="llama3.2:1b",
)

# Debugging: Check if 'Description' column exists
if 'Description' not in articles.columns:
    raise KeyError("The 'Description' column does not exist in the DataFrame.")

# Debugging: Check for missing or invalid data in 'Description'
if articles['Description'].is_null().any():
    raise ValueError("The 'Description' column contains missing values.")

# Generate embeddings for the 'Description' column in the articles DataFrame
embeddings = ollama_ef(articles['Description'].to_list())


In [10]:
collection.add(
    documents=articles['Description'].to_list(),
    ids=ids_list,
    embeddings=embeddings,
)

In [11]:
# Debugging: Verify the content of articles, ids_list, and embeddings
print("Sample articles:", articles.head())
print("Sample IDs:", ids_list[:5])
print("Sample embeddings:", embeddings[:5])

# Verify the collection's state
print(f"Collection contains {len(collection.get()['documents'])} documents.")

Sample articles: shape: (5, 5)
┌───────────┬─────────────┬──────────┬─────────────┬─────────────────────────────────┐
│ Defect_ID ┆ Category    ┆ Severity ┆ Status      ┆ Description                     │
│ ---       ┆ ---         ┆ ---      ┆ ---         ┆ ---                             │
│ str       ┆ str         ┆ str      ┆ str         ┆ str                             │
╞═══════════╪═════════════╪══════════╪═════════════╪═════════════════════════════════╡
│ D017      ┆ Security    ┆ Critical ┆ In Progress ┆ Failure to send notifications   │
│ D098      ┆ Performance ┆ Medium   ┆ Open        ┆ Inconsistent data display       │
│ D007      ┆ UI          ┆ Critical ┆ In Progress ┆ Slow response in data processi… │
│ D014      ┆ Performance ┆ Medium   ┆ Open        ┆ Failure to load external resou… │
│ D029      ┆ Hardware    ┆ High     ┆ Open        ┆ SSL certificate expired         │
└───────────┴─────────────┴──────────┴─────────────┴─────────────────────────────────┘
Sample IDs: 

In [ ]:
from pprint import pprint
query = "payment processing"
# Ensure input is a list for embedding function
query_embedding = ollama_ef([query])

# If the embedding function returns a list of embeddings, extract the first
if isinstance(query_embedding, list) and len(query_embedding) == 1:
    query_embedding = query_embedding[0]

print("Query embedding (first 5 values):", query_embedding[:5] if hasattr(query_embedding, '__getitem__') else query_embedding)

results = collection.query(
    query_embeddings=[query_embedding],  # Ensure this is a list of embeddings
    n_results=10,
)

print("Query results:")
pprint(results)

Query embedding: [array([-0.00800303,  0.03160303,  0.02629911, ...,  0.0069997 ,
       -0.05610436,  0.00725306], shape=(2048,), dtype=float32)]
Query results:
{'data': None,
 'distances': [[0.27019354701042175,
                0.5535998344421387,
                0.5746830701828003,
                0.5801409482955933,
                0.5924827456474304,
                0.6022247076034546,
                0.6309447884559631,
                0.6770452260971069,
                0.7199032306671143,
                0.723752498626709]],
 'documents': [['Error in payment processing',
                'Slow response in data processing',
                'Error in data import process',
                'Error in user authentication',
                'Incorrect data synchronization',
                'Error in file upload process',
                'Inconsistent data display',
                'Unexpected application shutdown',
                'Failure to send notifications',
                'Hard d